<a href="https://colab.research.google.com/github/rm571222/dataholics-oracle-challenge/blob/main/notebooks/02_data_upload/nb8_upload_populacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB8 — Carga no Oracle: População (T_SIH_MUNICIPIO, External Table)

**Projeto DATAHOLICS — FIAP Challenge | Parceria Oracle**

Este notebook documenta a última tabela do modelo — estimativas de população por município — carregada como **external table**, a terceira peça da arquitetura de 3 formatos do projeto (relacional + JSON + external table/CSV). Diferente das demais, o dado não é copiado para dentro do banco: o Oracle lê o arquivo diretamente do Object Storage, no lugar.

## Estrutura deste notebook
1. Preparação do CSV final
2. Upload para o Object Storage da Oracle
3. Criação da external table
4. Validação

## 1. Preparação do CSV final

O tratamento (junção de código IBGE, remoção de dígito verificador, filtro para SP) já foi validado na exploração (NB4). Aqui reproduzimos o tratamento e exportamos o CSV pronto para upload.

In [ ]:
import pandas as pd

def tratar_populacao_ibge(url, ano, nome_aba):
    raw = pd.read_excel(url, sheet_name=nome_aba, header=1)
    raw = raw.dropna(axis=1, how='all')
    raw = raw.dropna(subset=['COD. UF', 'COD. MUNIC'])

    raw['COD_MUNIC_7'] = (
        raw['COD. UF'].astype(int).astype(str).str.zfill(2) +
        raw['COD. MUNIC'].astype(int).astype(str).str.zfill(5)
    )
    raw['COD_MUNIC_6'] = raw['COD_MUNIC_7'].str[:-1]

    df_sp = raw[raw['UF'] == 'SP'].copy()
    df_sp = df_sp.rename(columns={'NOME DO MUNICÍPIO': 'NOME_MUNICIPIO', 'POPULAÇÃO ESTIMADA': 'POPULACAO'})
    df_sp['ANO_REF'] = ano

    return df_sp[['COD_MUNIC_6', 'COD_MUNIC_7', 'NOME_MUNICIPIO', 'POPULACAO', 'ANO_REF']]

df_pop_2024 = tratar_populacao_ibge(
    "https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2024/POP2024_20241230.xls", 2024, 'MUNICÍPIOS'
)
df_pop_2025 = tratar_populacao_ibge(
    "https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2025/POP2025_20260828.xls", 2025, 'Municípios'
)

df_populacao_sp = pd.concat([df_pop_2024, df_pop_2025], ignore_index=True)

COLUNAS_POPULACAO = ['COD_MUNIC_6', 'COD_MUNIC_7', 'NOME_MUNICIPIO', 'POPULACAO', 'ANO_REF']
df_populacao_sp[COLUNAS_POPULACAO].to_csv('populacao_municipios_sp.csv', index=False)

print(f"Total de linhas: {len(df_populacao_sp)}")

from google.colab import files
files.download('populacao_municipios_sp.csv')

## 2. Upload para o Object Storage da Oracle

Diferente das tabelas relacionais e do documento JSON (carregados via `INSERT` a partir do Colab), a external table exige que o arquivo esteja hospedado no Object Storage da OCI antes da criação da tabela. O caminho utilizado:

1. Criação de um bucket no Object Storage (console OCI → Storage → Buckets)
2. Upload manual do `populacao_municipios_sp.csv` para o bucket
3. Geração de uma **Pre-Authenticated Request (PAR)** — um link de acesso temporário com token de autorização embutido, que dispensa configuração de credencial adicional no banco (`DBMS_CLOUD.CREATE_CREDENTIAL`)

Esse caminho foi escolhido em vez de configurar uma credencial de API Key da OCI, por ser mais direto e não exigir geração de chaves adicionais para o escopo deste projeto.

## 3. Criação da external table

Executado no Database Actions (SQL Worksheet), usando a URL do PAR gerado na etapa anterior:

```sql
BEGIN
    DBMS_CLOUD.CREATE_EXTERNAL_TABLE(
        table_name => 'T_SIH_MUNICIPIO',
        file_uri_list => 'https://objectstorage.sa-saopaulo-1.oraclecloud.com/p/hdGxl3LEWLPu3hTDOxyjGpF3FbXqakpbE_dOsDEMsP2-3ThLfEdaXs3FwYec2Nx4/n/grl6lz9nu1rb/b/dataholics-bucket/o/populacao_municipios_sp.csv',
        format => JSON_OBJECT('type' value 'csv', 'skipheaders' value '1'),
        column_list => 'cd_municipio VARCHAR2(6),
                         cd_municipio_ibge VARCHAR2(7),
                         nm_municipio VARCHAR2(100),
                         qt_populacao NUMBER,
                         nr_ano_referencia NUMBER'
    );
END;
/
```

## 4. Validação

Confirmação de que a tabela foi criada como external table de fato (não uma tabela comum), e checagem de integridade contra a tabela fato.

```sql
-- Confirma que é external table
SELECT table_name, external FROM user_tables WHERE table_name = 'T_SIH_MUNICIPIO';
-- Resultado: EXTERNAL = 'YES'

-- Confirma o total de linhas
SELECT COUNT(*) FROM T_SIH_MUNICIPIO;
-- Resultado: 1290 (645 municípios x 2 anos)

-- Confirma que todo hospital da tabela fato tem população correspondente
SELECT COUNT(DISTINCT i.cd_municipio_hospital)
FROM T_SIH_INTERNACAO i
LEFT JOIN T_SIH_MUNICIPIO m ON i.cd_municipio_hospital = m.cd_municipio
WHERE m.cd_municipio IS NULL;
-- Resultado: 0
```

## Conclusão — NB8

A tabela `T_SIH_MUNICIPIO` foi criada como external table, com 1.290 registros (645 municípios de SP × 2 anos de referência) e 100% de cobertura contra os municípios de hospital da tabela fato. Com esta tabela, a arquitetura de 3 formatos do projeto está completa: **relacional** (`T_SIH_INTERNACAO`, `T_SIH_HOSPITAL` e as tabelas de domínio/CID-10/região), **documento JSON** (`T_SIH_ESTABELECIMENTO`) e **external table/CSV** (`T_SIH_MUNICIPIO`).